# Proposed validation — review before it runs

**What this measures:** The target, `n_coherent_factors`, counts how many of the 10 learned factors on the squidpy built-in Visium mouse-brain slice clear a permutation-null Moran's I z > 3 — the behavioral spatial-factor-recovery gate the requester named — while guardrails pin optimization health (relative loss drop on the 2-epoch synthetic smoke that replicates `tests/external/nsf/test_nsf.py`'s protocol), the NSF contract (non-negative, finite, correctly-shaped factors/loadings), and non-degeneracy (min factor CV). This is a one-arm capability question because `main` cannot import `scvi.external.NSF` at all, so no baseline arm and no sentinel numbers; the run must execute under the repo's Python ≥ 3.12 environment, which the eval image provides (the requester's local 3.10 cannot).

**Target metric:** `n_coherent_factors`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

This notebook is the only copy of the test: `.remyx/validation.yaml` points here, and the run executes these cells top to bottom.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

## Execution context

The cells below are the test Remyx drafted, split into cells; it was written as a script, so `__file__` is set to `eval/eval_nsf_behavioral.py` (no such file is committed). This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_nsf_behavioral.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
"""Behavioral validation of the clean-room NSF implementation (scvi.external.NSF).

Two arms: on the PR head, the symbol added by the diff exists and the full behavioral measurement
runs. On the baseline (pre-change branch) `from scvi.external import NSF` fails cleanly via the
defensive import below and the script prints the SAME JSON keys with degraded/zeroed metrics —
there is no NSF to measure without the diff, so every count guardrail is vacuously 0 and the
target is 0 (that zero-vs-measured delta is the capability signal).

Phase 1 (user_guidance "CPU smoke"): synthetic AnnData + obsm["spatial"] + counts, setup_anndata,
train(max_epochs=2, accelerator="cpu") -> finite decreasing ELBO, finite non-negative factors/loadings
with the right shapes (reported as metric values, not asserts; thresholds live in validation.yaml).

Phase 2 (user_guidance "real behavioral gate"): squidpy built-in Visium mouse-brain slice at model
defaults (L=10, M=min(100,N), lengthscale_init=0.5); spatial coherence of each factor is a Moran's I
permutation z on a k=6 spot graph over standardized coords. Clean-room: no numerical parity vs the
LGPL/TFP reference, so the gate is behavioral only.

REMYX_SMOKE=1 runs the same code paths and JSON keys with ~500 spots / 500 HVGs / 2 Visium epochs /
20 permutations; those numbers are meaningless by design.
"""
import json
import math
import os
import sys
import tempfile
from pathlib import Path

In [ ]:
import numpy as np
import scanpy as sc
import squidpy
from anndata import AnnData
from scipy import sparse
from scipy.spatial import cKDTree

In [ ]:
_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(_ROOT / "src" if (_ROOT / "src" / "scvi").is_dir() else _ROOT))
from scvi import settings  # pre-existing symbol, present on both arms

try:  # symbol ADDED by the diff: absent on baseline -> fail cleanly, degraded metrics (rubric item 8)
    from scvi.external import NSF
except ImportError:
    NSF = None

In [ ]:
SMOKE = os.environ.get("REMYX_SMOKE", "0") == "1"
N_PERMS, N_SPOTS, N_HVGS, EPOCHS = (20, 500, 500, 2) if SMOKE else (100, None, 2000, 300)
N_FACTORS = 10  # NSF default (held constant); factor/loading shapes are checked exactly against it
BAD = -1e9  # non-finite floats fail every max-direction threshold

def _f(x):
    x = float(x)
    return x if math.isfinite(x) else BAD

In [ ]:
def _elbo(model):
    """Per-epoch training ELBO from the repo's own history dict (higher = better)."""
    return model.history["elbo_train"].iloc[:, 0].to_numpy(dtype=float)

def _rel_drop(elbo):
    """Relative loss drop over training: loss=-ELBO, so drop=(elbo_last-elbo_first)/|elbo_first|."""
    first, last = elbo[0], elbo[-1]
    if not (np.isfinite(first) and np.isfinite(last)) or first == 0:
        return BAD
    return (last - first) / abs(first)

In [ ]:
def _contract(F, W, n_obs, n_vars, stats):
    """Finiteness / non-negativity / exact shape contract: factors (N,L), loadings (L,G)."""
    stats["n_nonfinite"] += int((~np.isfinite(F)).sum() + (~np.isfinite(W)).sum())
    stats["n_negative_values"] += int((F < 0).sum() + (W < 0).sum())
    stats["n_shape_mismatches"] += int(F.shape != (n_obs, N_FACTORS)) + int(W.shape != (N_FACTORS, n_vars))

def _morans_z(F, coords, n_perms, seed=0):
    """One-sided Moran's I permutation z per factor on a symmetric binary k=6 kNN spot graph.

    Null E[I] = -1/(N-1) ~ 0; z > 3 is one-sided p ~ 1.35e-3 per factor, so >=5 coherent of 10
    (expected chance count ~0.0135) is chance-impossible — the behavioral bar from claim_analysis."""
    N, L = F.shape
    coords = (coords - coords.mean(0)) / coords.std(0)
    _, nbrs = cKDTree(coords).query(coords, k=min(7, N))
    rows = np.repeat(np.arange(N), nbrs.shape[1])
    A = sparse.coo_matrix((np.ones(rows.size), (rows, nbrs.ravel())), shape=(N, N))
    A = A.maximum(A.T).astype(np.float64).tocsr()
    w_sum = float(A.sum())
    Zc = F - F.mean(0)
    i_obs = (N / w_sum) * np.einsum("ij,ij->j", Zc, A @ Zc) / np.maximum((Zc ** 2).sum(0), 1e-30)
    rng = np.random.default_rng(seed)
    perms = np.stack([rng.permutation(N) for _ in range(n_perms)], axis=1)  # (N, R)
    Zp = Zc[perms]  # (N, R, L)
    AZp = (A @ Zp.reshape(N, -1)).reshape(N, n_perms, L)
    i_null = (N / w_sum) * np.einsum("nrj,nrj->jr", Zp, AZp) / np.maximum((Zp ** 2).sum(0).T, 1e-30)
    mu, sd = i_null.mean(1), i_null.std(1, ddof=1)
    return np.where(sd > 1e-12, (i_obs - mu) / np.maximum(sd, 1e-12), 0.0)

In [ ]:
def _degraded():
    """Baseline arm (pre-change branch): NSF absent, nothing to train — same keys, degraded values.

    Every count guardrail is vacuously 0 (no NSF outputs exist to be non-finite / negative /
    misshapen / constant / non-decreasing), so the guardrails stay satisfiable on baseline; the
    target is 0, which is exactly the pre-change capability."""
    return {
        "n_coherent_factors": 0,
        "smoke_loss_drop_failures": 0,
        "n_constant_factors": 0,
        "n_nonfinite": 0,
        "n_negative_values": 0,
        "n_shape_mismatches": 0,
        "mean_morans_z": 0.0,
        "smoke_loss_rel_drop": 0.0,
        "min_factor_cv": 0.0,
        "visium_loss_rel_drop": 0.0,
    }

In [ ]:
def main():
    settings.logging_dir = tempfile.mkdtemp(prefix="nsf_eval_")
    if NSF is None:  # baseline arm: defensive import failed -> degraded metrics, no crash
        print(json.dumps(_degraded()))
        return
    stats = {"n_nonfinite": 0, "n_negative_values": 0, "n_shape_mismatches": 0}

    # ---- phase 1: synthetic CPU smoke (repo test protocol) ----
    settings.seed = 0  # scvi seeds torch/numpy/random from this before the fit
    rng = np.random.default_rng(0)
    n_s, g_s = 200, 100
    syn = AnnData(
        X=rng.poisson(0.5, (n_s, g_s)).astype(np.float32),
        obsm={"spatial": rng.uniform(0.0, 2.0, (n_s, 2)).astype(np.float32)},
    )
    NSF.setup_anndata(syn, spatial_key="spatial")
    m = NSF(syn)  # defaults: n_factors=10, n_inducing_points=min(100,N), lengthscale_init=0.5
    m.train(max_epochs=2, accelerator="cpu", enable_progress_bar=False)
    Fs = np.asarray(m.get_factors(), dtype=np.float64)
    Ws = np.asarray(m.get_loadings(), dtype=np.float64)
    elbo_s = _elbo(m)
    stats["n_nonfinite"] += int((~np.isfinite(elbo_s)).sum())
    _contract(Fs, Ws, n_s, g_s, stats)
    rel_s = _rel_drop(elbo_s)
    # guardrail: 1 if the smoke ELBO failed to rise by >=0.1% relative (loss did not decrease)
    smoke_drop_failures = int(not (rel_s >= 0.001))

    # ---- phase 2: real Visium behavioral gate (squidpy built-in, cached fetch) ----
    settings.seed = 0  # re-seed so phase 2 is independent of phase-1 RNG consumption
    cache = Path(tempfile.gettempdir()) / "nsf_eval_visium"
    cache.mkdir(parents=True, exist_ok=True)
    adata = squidpy.datasets.visium(path=str(cache))  # V1_Mouse_Brain_Sagittal_Anterior, ~3355 spots
    if N_SPOTS is not None:
        idx = np.sort(np.random.default_rng(0).choice(adata.n_obs, N_SPOTS, replace=False))
        adata = adata[idx].copy()
    sc.pp.highly_variable_genes(adata, n_top_genes=N_HVGS, flavor="cell_ranger", subset=True)
    NSF.setup_anndata(adata, spatial_key="spatial")
    model = NSF(adata)
    model.train(max_epochs=EPOCHS, accelerator="cpu", enable_progress_bar=False)
    F = np.asarray(model.get_factors(), dtype=np.float64)
    Wl = np.asarray(model.get_loadings(), dtype=np.float64)
    elbo_v = _elbo(model)
    stats["n_nonfinite"] += int((~np.isfinite(elbo_v)).sum())
    _contract(F, Wl, adata.n_obs, adata.n_vars, stats)
    z = _morans_z(F, np.asarray(adata.obsm["spatial"], dtype=np.float64), N_PERMS)
    # CV < 1e-4 == numerically constant factor: F=exp(f) is O(1) and float32 eps is 1.2e-7
    cv = F.std(0) / np.maximum(F.mean(0), 1e-30)
    n_constant_factors = int((cv < 1e-4).sum())

    print(json.dumps({
        "n_coherent_factors": int((z > 3).sum()),
        "smoke_loss_drop_failures": smoke_drop_failures,
        "n_constant_factors": n_constant_factors,
        "n_nonfinite": int(stats["n_nonfinite"]),
        "n_negative_values": int(stats["n_negative_values"]),
        "n_shape_mismatches": int(stats["n_shape_mismatches"]),
        "mean_morans_z": _f(z.mean()),
        "smoke_loss_rel_drop": _f(rel_s),
        "min_factor_cv": _f(cv.min()),
        "visium_loss_rel_drop": _f(_rel_drop(elbo_v)),
    }))

if __name__ == "__main__":
    main()  # CLI flags (--variant/--ref/--seed) are accepted and ignored; failures raise with traceback

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "Does the clean-room NSF (scvi.external.NSF) train sensibly on CPU — finite, decreasing loss and finite, non-negative factors/loadings of the right shapes — and behaviorally recover spatially-coherent non-negative factors on a real Visium slice?"
loop:
  max_iterations: 8
  fix_code: true
benchmarks:
  - name: nsf-behavioral-cpu
    suite: "eval/eval_nsf_behavioral.py"
    # script honors REMYX_SMOKE=1: same code paths and JSON keys, 2-epoch Visium fit on ~500 spots / 500 HVGs / 20 permutations
    # two arms: baseline (main) has no scvi.external.NSF -> defensive ImportError -> same JSON keys with degraded
    # (zeroed) metrics and no crash; head arm runs the full behavioral measurement
    packages: ["squidpy"]
    scorer: n_coherent_factors
    baseline: main  # pre-change branch: scvi.external.NSF absent -> defensive import -> degraded metrics
    policy:
      guardrail_veto: true
    metrics:
      - name: n_coherent_factors
        direction: max
        threshold: 5
        role: target
        bar: goal
        reads_as: "of 10 learned Visium factors, how many are spatially coherent (Moran's I permutation z > 3)"
        # derivation: per-factor one-sided permutation p(z>3) ~ 1.35e-3; expected chance count across 10
        # factors ~ 0.0135, so >=5 (ceil(L/2), claim_analysis) is chance-impossible and matches the paper's
        # qualitative claim that most Visium-brain factors sit at layer/domain scale.
        # baseline arm prints 0 (no NSF exists pre-change) — the target is the capability delta, not a guardrail.
      - name: smoke_loss_drop_failures
        direction: min
        threshold: 0
        role: guardrail
        bar: floor
        reads_as: "1 if the 2-epoch synthetic smoke ELBO failed to rise by >=0.1% relative (loss did not decrease) or was non-finite, else 0"
        # derivation: float32 relative noise ~1.2e-7*|loss| while healthy first Adam epochs drop tens of
        # percent, so a 1e-3 relative-rise floor separates 'optimizer moved' from 'numeric tie'
        # (claim_analysis arithmetic); encoded as a 0/1 failure count so the baseline arm (no NSF -> no
        # smoke to fail) vacuously satisfies it with 0, per the baseline-achievable guardrail rule.
      - name: n_constant_factors
        direction: min
        threshold: 0
        role: guardrail
        bar: floor
        reads_as: "count of Visium factors with variability (std/mean) below 1e-4, i.e. numerically constant"
        # derivation: F=exp(f) with zero-mean GP prior is O(1); float32 eps 1.19e-7, so CV<1e-4 == flat field.
        # failure-count form keeps the baseline arm at 0 (no factors exist pre-change).
      - name: n_nonfinite
        direction: min
        threshold: 0
        role: guardrail
        bar: sanity
        reads_as: "count of non-finite entries among factors, loadings and recorded losses"
        # baseline prints 0 (nothing ran), so the pre-change code satisfies it; fails on any NaN/inf.
      - name: n_negative_values
        direction: min
        threshold: 0
        role: guardrail
        bar: sanity
        reads_as: "count of negative entries among factors and loadings (must be exactly zero)"
        # baseline prints 0; fails the moment a factor/loading entry is < 0.
      - name: n_shape_mismatches
        direction: min
        threshold: 0
        role: guardrail
        bar: sanity
        reads_as: "count of exact shape mismatches vs (n_spots, 10) factors and (10, n_genes) loadings"
        # baseline prints 0; fails if get_factors/get_loadings return any other shape (checked exactly).
      - name: mean_morans_z
        direction: max
        threshold: 3
        role: diagnostic
        reads_as: "mean Moran's I permutation z across all 10 factors (informational; 0.0 on the baseline arm)"
      - name: smoke_loss_rel_drop
        direction: max
        threshold: 0.001
        role: diagnostic
        reads_as: "relative training-loss drop over the 2-epoch synthetic smoke (informational float behind the guardrail; 0.0 on baseline)"
      - name: min_factor_cv
        direction: max
        threshold: 0.0001
        role: diagnostic
        reads_as: "smallest factor variability std/mean (informational float behind the guardrail; 0.0 on baseline)"
      - name: visium_loss_rel_drop
        direction: max
        threshold: 0.01
        role: diagnostic
        reads_as: "relative training-loss drop from first to last epoch of the full Visium fit (0.0 on baseline)"
    compute:
      tier: cpu
      # budget: 300 epochs x ~27 steps/epoch (3355 spots / batch 128) x O(batch*2000 Poisson + batch*100*10 GP)
      # ~ seconds/epoch => ~15-30 min, + ~150 MB Visium fetch + 2-epoch smoke + 100-perm Moran (seconds)
      timeout_s: 5400
    held_constant:
      - "seed: scvi.settings.seed=0 re-set before each fit plus fixed numpy rng seeds for synthetic counts, coords, subsample and Moran permutations"
      - "dataset: squidpy.datasets.visium() default V1_Mouse_Brain_Sagittal_Anterior slice (~3355 spots), cached fetch"
      - "preprocessing: raw counts, top 2000 HVGs (scanpy, cell_ranger), k=6 kNN graph on standardized coords for Moran's I"
      - "model defaults: n_factors=10, n_inducing_points=min(100,N), lengthscale_init=0.5, loadings_scale=1.0"
      - "expected shapes: factors exactly (n_spots, 10) and loadings exactly (10, n_genes)"
      - "training: max_epochs=2 synthetic smoke and 300 Visium epochs, accelerator='cpu', no early stopping"
      - "env: python >=3.12 per repo requires-python (requester's local 3.10 cannot run this scvi-tools)"
    avoid:
      - "no numerical ELBO/factor parity vs the LGPL/TFP reference (clean-room; user explicitly waives parity)"
      - "no wall-clock or throughput metrics on shared CPU; the claim is not about the device"
      - "do not gate on raw Moran's I (near-constant factors inflate it); the gate is permutation z>3 plus the n_constant_factors non-constancy guardrail"
      - "no unpinned revisions or live-API results; only the squidpy built-in Visium fetch and the repo's own synthetic protocol"
      - "do not swallow dataset-loading, model-construction, training or scoring errors with blanket try/except: only the added-symbol import (scvi.external.NSF) is defensive (ImportError -> degraded baseline arm); every other failure must raise with a traceback"
    provenance:
      n_coherent_factors: "user_guidance"
      smoke_loss_drop_failures: "user_guidance"
      smoke_loss_rel_drop: "user_guidance"
      n_nonfinite: "user_guidance"
      n_negative_values: "user_guidance"
      n_shape_mismatches: "user_guidance"
      n_constant_factors: "inferred"
      mean_morans_z: "inferred"
      min_factor_cv: "inferred"
      visium_loss_rel_drop: "inferred"
      suite: "synthesized"
      held_constant: "user_guidance"
      compute: "user_guidance"
report:
  headline: "Clean-room NSF trains sensibly on CPU and recovers a majority of spatially-coherent non-negative factors on Visium mouse brain (behavioral validation, no reference parity)"
  findings:
    - "{n_coherent_factors} of 10 factors are spatially coherent (Moran's I permutation z > 3) on the Visium slice"
    - "the 2-epoch synthetic smoke loss dropped {smoke_loss_rel_drop:%} ({smoke_loss_drop_failures} drop failures) with {n_nonfinite} non-finite and {n_negative_values} negative factor/loading values and {n_shape_mismatches} shape mismatches"
    - "{n_constant_factors} Visium factors are numerically constant (smallest factor CV is {min_factor_cv}), so no factor collapsed to a flat field"
    - "mean Moran's z across all factors is {mean_morans_z} and the full Visium fit loss dropped {visium_loss_rel_drop:%} from first to last epoch"
  establishes:
    - "the SVGP + inducing-point ELBO optimizes sensibly: finite decreasing loss on synthetic and real data (rests on {smoke_loss_drop_failures} and {visium_loss_rel_drop})"
    - "the model honors its contract: non-negative, finite, correctly-shaped factors and loadings (rests on {n_negative_values}, {n_nonfinite}, {n_shape_mismatches})"
    - "behavioral spatial-factor recovery qualitatively matching Townes & Engelhardt, without reference parity (rests on {n_coherent_factors} with the {n_constant_factors} non-degeneracy guardrail)"
  does_not_establish:
    - "numerical parity with the authors' LGPL/TFP reference implementation (waived: clean-room)"
    - "the paper's quantitative metrics: held-out RMSE vs MEFISTO/PCA/NMF/GLM-PCA, simulation factor-recovery error, cross-slice generalization, NSFH spatial importances"
    - "anatomical identity of factors (cortical layers, hippocampus): coherence here is statistical, not label-matched"
  not_measured:
    - "held-out count prediction RMSE"
    - "runtime/scalability vs a full GP (O(NM^2)); wall-clock omitted on shared CPU"
    - "hybrid NSFH and real-valued RSF variants"
  caveat: "Clean-room PyTorch reimplementation vs the paper's TFP reference with a different (CVI-style) optimizer path, so ELBO values and trajectories are not comparable; all thresholds are behavioral. The baseline arm has no NSF at all and reports degraded (zeroed) metrics by design."
  next: "Add a simulation with planted spatial factors to measure factor-recovery correlation, plus a second Visium section for cross-slice generalizability."
```